# Function Manager

## Component: Conv2D layer

### CPU

In [ ]:
def conv2d_cpu(input4d, kernel4d, bias1d, output4d, stride, padding, dilation):
  for im in range(output4d.shape[0]):
    for c_out in range(output4d.shape[1]):
      for x in range(output4d.shape[2]):
        for y in range(output4d.shape[3]):
          sum = 0
          for x_k in range(kernel4d.shape[2]):
            for y_k in range(kernel4d.shape[3]):
              x_in = x * stride + x_k * dilation - padding
              y_in = y * stride + y_k * dilation - padding
              if (0 <= x_in < input4d.shape[2]) and (0 <= y_in < input4d.shape[3]):
                for c_in in range(input4d.shape[1]):
                  sum += input4d[im, c_in, x_in, y_in] * kernel4d[c_out, c_in, x_k, y_k]
          output4d[im, c_out, x, y] = sum + bias1d[c_out]

# def conv2d_biasBack_cpu(d_output4d, bias1d):
#   for c_out in range(bias1d.shape[0]):
#     sum = 0
#     for im in range(d_output4d.shape[0]):
#       for x in range(d_output4d.shape[2]):
#         for y in range(d_output4d.shape[3]):
#           sum += d_output4d[im, c_out, x, y]
#     bias1d[c_out] = sum

# def flipKernel_cpu(kernel4d, output4d):
#   for c_out in range(kernel4d.shape[0]):
#     for c_in in range(kernel4d.shape[1]):
#       for x_k in range(kernel4d.shape[2]):
#         for y_k in range(kernel4d.shape[3]):
#           output4d[c_out, c_in, x_k, y_k] = kernel4d[c_out, c_in, kernel4d.shape[2]-x_k-1, kernel4d.shape[3]-y_k-1]

### GPU

In a convolution process, the convolution calculation is obviously most used. However, for the sake of simplifying the backward process, we flip the kernel before calculating convolution, instead of combining both.

Also, during the backward process, we calculate the bias gradient by taking sum per channel dimension.

#### Convolution 2D
Input size: $[N, C_{in}, H_{in}, W_{in}]$, kernel size: $[C_{cout}, C_{in}, K, K]$, bias size: $[C_{out}]$, output size: $[N, C_{out}, H_{out}, W_{out}]$.

Here is our strategy:

1. Divide inputs and outputs by $N$ and $C\_{out}$ dimensions (through CUDA streams): because CUDA cannot deal with over 3 dimensions, and these dimensions easily stand out. We do not split up to $C\_{in}$ since the number of parameters become too small. # of input parameters estimated to be around 500 x 400 x 400 on average.
2. Per channel, convolution is like multiplication between 2D matrices, so we can implement such. Memory coalescing suggests that we should store memory on GMEM to be consecutive, so we shall have 2 versions of this: $[H, W, C]$ and $[C, H, W]$.
3. Let's say we tile with size $[C, 32, 32]$, we would still add 1 to the final dimension to avoid bank conflicts.
4. Finally, max kernel size this way is 1024 x 3 x 3 = 9K numbers, which is 36KB if we use float32. CMEM has 64KB, so we can use this... Unfortunately, Numba does not support applying array that could be changed during runtime so... we use SMEM.

In [ ]:
!uv pip install -q --system numba-cuda==0.4.0
from numba import config
config.CUDA_ENABLE_PYNVJITLINK = 1

In [ ]:
from numba import cuda
import numpy as np

TILE_SIZE = 16
@cuda.jit
def conv2d_gpu_v1(input, kernel, bias, w, h, c, kernel_size, stride, pad, dilation, output):
  inputS = cuda.shared.array((18, 19), np.float32)
  kernelS = cuda.shared.array((3, 3), np.float32)

  tx = cuda.threadIdx.x
  ty = cuda.threadIdx.y
  dx = cuda.blockDim.x
  dy = cuda.blockDim.y
  bx = cuda.blockIdx.x
  by = cuda.blockIdx.y
  col_o = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
  row_o = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
  col_i = col_o - pad
  row_i = row_o - pad

  # imagine smem has size 2*TILE_SIZE, that we assign each thread to take care 4.
  sum = 0
  for z in range(c):
    for i in range(2):
      for j in range(2):
          x = tx + i * TILE_SIZE
          y = ty + j * TILE_SIZE
          global_x = bx * TILE_SIZE + x - pad
          global_y = by * TILE_SIZE + y - pad
          if x < 18 and y < 18:
            if 0 <= global_x < w and 0 <= global_y < h:
                inputS[x, y] = input[z, global_x, global_y]
            else:
                inputS[x, y] = 0.0
    if tx < kernel_size and ty < kernel_size:
      kernelS[tx, ty] = kernel[z, tx, ty]
    cuda.syncthreads()

    if row_o < h and col_o < w:
      for x_k in range(kernel_size):
        for y_k in range(kernel_size):
          # (tx, ty) = (0, 0) -> (2, 2)
          in_x = tx * stride + x_k * dilation # remember, tx is the start of left and upmost.
          in_y = ty * stride + y_k * dilation
          if 0 <= in_x < 18 and 0 <= in_y < 18:
            sum += inputS[in_x, in_y] * kernelS[x_k, y_k]
    cuda.syncthreads()

  output[col_o, row_o] = sum + bias

In [ ]:
input2d = np.random.randn(3, 8, 128, 128).astype(np.float32)
kernel2d = np.random.randn(4, 8, 3, 3).astype(np.float32)
output2d_gpu = np.zeros((3, 4, 128, 128), dtype=np.float32)
output2d = np.zeros((3, 4, 128, 128), dtype=np.float32)
bias1d = np.random.randn(4).astype(np.float32)
stride, pad, dilation = 1, 1, 1

blockSize = (16, 16)
gridSize = (8, 8)

from numba import cuda
streams = [[cuda.stream() for _ in range(4)] for _ in range(3)]
for i in range(3):
  for j in range(4):
    input = input2d[i, :, :, :]
    kernel = kernel2d[j, :, :, :]
    bias = bias1d[j]

    d_input = cuda.to_device(input, stream=streams[i][j])
    d_kernel = cuda.to_device(kernel, stream=streams[i][j])
    d_output = cuda.device_array((128, 128), dtype=np.float32, stream=streams[i][j])

    conv2d_gpu_v1[gridSize, blockSize, streams[i][j]](d_input, d_kernel, bias, 128, 128, 8, 3, 1, 1, 1, d_output)

    d_output.copy_to_host(output2d_gpu[i][j], stream=streams[i][j])

cuda.synchronize()
conv2d_cpu(input2d, kernel2d, bias1d, output2d, stride, pad, dilation)

print(np.mean(np.abs(output2d - output2d_gpu)))
print(np.var(np.abs(output2d - output2d_gpu)))

/usr/local/lib/python3.11/dist-packages/numba_cuda/numba/cuda/dispatcher.py:605: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.11/dist-packages/numba_cuda/numba/cuda/dispatcher.py:605: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


8.675176e-07
8.5233876e-13


In [ ]:
from numba import cuda
import numpy as np

TILE = 16  # Safe for shared memory
C_IN = 128
@cuda.jit
def conv2d_gpu_v2(input, kernel, bias, output,
                  N, C_in, C_out, H, W, K, stride, pad, dilation):
    # Shared memory tile with padding
    inputS = cuda.shared.array((TILE + 2, TILE + 2), dtype=np.float32)
    kernelS = cuda.shared.array((C_IN, 3, 3), dtype=np.float32)  # Could be optimized

    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    bx = cuda.blockIdx.x
    by = cuda.blockIdx.y
    bz = cuda.blockIdx.z  # Maps to (n, c_out)
    dx = cuda.blockDim.x
    dy = cuda.blockDim.y

    # Decode (n, c_out) from blockIdx.z
    n = bz // C_out
    c_out = bz % C_out

    # Compute output pixel (y, x)
    out_x = bx * dx + tx
    out_y = by * dy + ty

    if out_x >= W or out_y >= H:
        return

    acc = 0.0
    for c_in in range(C_in):
        # Preload kernel weights (1 thread block-wide copy)
        if tx < K and ty < K:
            kernelS[c_in, tx, ty] = kernel[c_out, c_in, tx, ty]
        cuda.syncthreads()

        # Compute input coords
        in_x = out_x * stride - pad
        in_y = out_y * stride - pad

        for i in range(K):
            for j in range(K):
                xi = in_x + i * dilation
                yj = in_y + j * dilation

                if 0 <= xi < W and 0 <= yj < H:
                    acc += input[n, c_in, yj, xi] * kernelS[c_in, i, j]
    output[n, c_out, out_y, out_x] = acc + bias[c_out]

N, C_in, H, W = 8, 128, 416, 416
C_out, K = 1024, 3
stride, pad, dilation = 1, 1, 1

input2d = np.random.randn(N, C_in, H, W).astype(np.float32)
kernel2d = np.random.randn(C_out, C_in, K, K).astype(np.float32)
bias1d = np.random.randn(C_out).astype(np.float32)
output2d_gpu = np.zeros((N, C_out, H, W), dtype=np.float32)

# Copy to device
d_input = cuda.to_device(input2d)
d_kernel = cuda.to_device(kernel2d)
d_bias = cuda.to_device(bias1d)
d_output = cuda.device_array((N, C_out, H, W), dtype=np.float32)

# Launch config
blockSize = (TILE, TILE)
import math
gridSize = (math.ceil(W / TILE), math.ceil(H / TILE), N * C_out)

# Launch kernel
conv2d_gpu_v2[gridSize, blockSize](
    d_input, d_kernel, d_bias, d_output,
    N, C_in, C_out, H, W, K, stride, pad, dilation
)

# Copy back
d_output.copy_to_host(output2d_gpu)

TypingError: Failed in cuda mode pipeline (step: nopython frontend)
No implementation of function Function(<function shared.array at 0x7e31bf2be340>) found for signature:
 
 >>> array(UniTuple(int64 x 2), dtype=class(float32))
 
There are 2 candidate implementations:
  - Of which 2 did not match due to:
  Overload of function 'array': File: numba_cuda/numba/cuda/cudadecl.py: Line 27.
    With argument(s): '(UniTuple(int64 x 2), dtype=class(float32))':
   No match.

During: resolving callee type: Function(<function shared.array at 0x7e31bf2be340>)
During: typing of call at /tmp/ipython-input-1958927362.py (10)


File "../tmp/ipython-input-1958927362.py", line 10:
<source missing, REPL/exec in use?>


In [ ]:
del d_input, d_kernel, d_bias, d_output

In [ ]:
# import numpy as np
# from numba import cuda, float32

# @cuda.jit
# def conv2d_kernel(input, kernel, output):
#     i, j = cuda.grid(2)
#     k_h, k_w = kernel.shape
#     i_h, i_w = input.shape

#     # Only compute if within bounds of output
#     if i < output.shape[0] and j < output.shape[1]:
#         acc = 0.0
#         for ki in range(k_h):
#             for kj in range(k_w):
#                 acc += input[i + ki, j + kj] * kernel[ki, kj]
#         output[i, j] = acc

# # Input size: (6x6), Kernel size: (3x3), Output size: (4x4)
# input_cpu = np.random.rand(6, 6).astype(np.float32)
# kernel_cpu = np.random.rand(3, 3).astype(np.float32)
# output_cpu = np.zeros((4, 4), dtype=np.float32)

# # Allocate and copy to device
# input_gpu = cuda.to_device(input_cpu)
# kernel_gpu = cuda.to_device(kernel_cpu)
# output_gpu = cuda.device_array((4, 4), dtype=np.float32)

# # Define grid and block dimensions
# threadsperblock = (16, 16)
# blockspergrid_x = (output_cpu.shape[0] + threadsperblock[0] - 1) // threadsperblock[0]
# blockspergrid_y = (output_cpu.shape[1] + threadsperblock[1] - 1) // threadsperblock[1]
# blockspergrid = (blockspergrid_x, blockspergrid_y)

# # Launch kernel
# conv2d_kernel[blockspergrid, threadsperblock](input_gpu, kernel_gpu, output_gpu)

# # Copy result back
# output_cpu = output_gpu.copy_to_host()

# # Print results
# print("Input:\n", input_cpu)
# print("Kernel:\n", kernel_cpu)
# print("Output (Valid Conv2D):\n", output_cpu)
